# GRPO fine-tuning of Gemma-3-1B on PokerBench (Colab, T4 GPU)

Pipeline: **SFT warm-start -> GRPO with a verifiable solver-based reward -> evaluation**.
Reward/eval code lives in the repo (`logicalllm/poker/`) and is unit-tested; this notebook only wires it to Unsloth.

> Runtime -> Change runtime type -> **T4 GPU**. No secrets are needed (PokerBench and Gemma-3 via `unsloth/` mirror are public). Don't paste tokens into notebooks.

In [ ]:
%%capture
!pip install unsloth
!git clone https://github.com/nishilseth/LogicalLLM.git
%cd LogicalLLM

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it", max_seq_length=max_seq_length, load_in_4bit=False, full_finetuning=False)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias="none", random_state=3407,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

In [ ]:
from logicalllm.poker.data import load_pokerbench, to_chat
from logicalllm.poker.reward import grpo_reward_fn, poker_reward, MAX_REWARD
from logicalllm.poker.evaluate import score

train = load_pokerbench("train", n=6000, seed=0)      # class-balanced subsample
test  = load_pokerbench("test",  n=500,  seed=123)    # fixed eval set (same as the local experiments)
print(train[0]["instruction"][-400:], "->", train[0]["output"])

## 0. Zero-shot baseline

In [ ]:
def predict(prompts, bs=16):
    FastLanguageModel.for_inference(model)
    outs = []
    for i in range(0, len(prompts), bs):
        texts = [tokenizer.apply_chat_template(to_chat(p), tokenize=False, add_generation_prompt=True) for p in prompts[i:i+bs]]
        tokenizer.padding_side = "left"
        enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, max_new_tokens=12, do_sample=False)
        outs += [o.strip() for o in tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return outs

base = score(predict(test["instruction"]), test["output"]); print("zero-shot:", base)

## 1. SFT warm-start
GRPO needs *some* reward variance inside each group; a model that never emits a valid action gets zero signal. A short SFT pass teaches the output format.

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
sft_ds = Dataset.from_list([{"messages": to_chat(i) + [{"role":"assistant","content":o}]}
                            for i, o in zip(train["instruction"], train["output"])])
FastLanguageModel.for_training(model)
SFTTrainer(model=model, processing_class=tokenizer, train_dataset=sft_ds,
           args=SFTConfig(per_device_train_batch_size=4, gradient_accumulation_steps=4, max_steps=150, learning_rate=2e-4,
                          logging_steps=10, output_dir="outputs/sft", report_to="none", assistant_only_loss=False,
                          max_length=max_seq_length, fp16=True)).train()
after_sft = score(predict(test["instruction"]), test["output"]); print("after SFT:", after_sft)

## 2. GRPO
For each prompt sample a **group** of G actions, score each with `poker_reward`, and use the z-scored reward inside the group as the advantage:
`A_i = (r_i - mean(r)) / std(r)`. No value network. Clipped PPO-style update, optional KL penalty (`beta`).

In [ ]:
from trl import GRPOConfig, GRPOTrainer
grpo_ds = train.select(range(1500)).map(lambda r: {"prompt": to_chat(r["instruction"]), "gold": r["output"]},
                                        remove_columns=train.column_names)
FastLanguageModel.for_training(model)
cfg = GRPOConfig(output_dir="outputs/grpo", num_generations=8, per_device_train_batch_size=8, gradient_accumulation_steps=2,
                 max_completion_length=12, max_steps=300, learning_rate=5e-6, beta=0.02, temperature=1.0,
                 logging_steps=1, report_to="tensorboard", logging_dir="outputs/tb", fp16=True, remove_unused_columns=False)
trainer = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=grpo_reward_fn, args=cfg, train_dataset=grpo_ds)
trainer.train()

In [ ]:
after_grpo = score(predict(test["instruction"]), test["output"]); print("after GRPO:", after_grpo)
import json; json.dump({"zero_shot": base, "sft": after_sft, "grpo": after_grpo}, open("colab_results.json", "w"), indent=1)

## 3. Training curves

In [ ]:
import matplotlib.pyplot as plt
h = [x for x in trainer.state.log_history if "reward" in x]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot([x["step"] for x in h], [x["reward"] for x in h]); ax[0].set_title("mean reward")
if "kl" in h[0]: ax[1].plot([x["step"] for x in h], [x["kl"] for x in h]); ax[1].set_title("KL to reference")
ax[2].plot([x["step"] for x in h], [x.get("reward_std", 0) for x in h]); ax[2].set_title("reward std within group")
plt.tight_layout(); plt.savefig("grpo_curves.png", dpi=140)

In [ ]:
model.save_pretrained("poker-gemma3-grpo-lora")   # LoRA adapter; download or push_to_hub with your own login